# Semantic Kernel Chat Completion Agent

Chat Completion is fundamentally a protocol for a chat-based interaction with an AI model where the chat-history maintained and presented to the model with each request. Semantic Kernel AI services offer a unified framework for integrating the chat-completion capabilities of various AI models.

A chat completion agent can leverage any of these AI services to generate responses, whether directed to a user or another agent.


## Creating the required Azure resources

You can run the powershell script `create-azure-ai-resources.ps1` to create the following resources:
* Azure AI Services with Hub and Project
* GPT-4o model

## Installing the necessary packages

Import Semantic Kernel SDK from pypi.org

In [1]:
%pip install -U semantic-kernel --quiet

Note: you may need to restart the kernel to use updated packages.


Check the current version of Semantic Kernel.

In [3]:
from semantic_kernel import __version__

__version__

'1.29.0'

In [4]:
from dotenv import load_dotenv
import os

if os.path.exists(".env"):
    load_dotenv(override=True)

## Creating a chat completion service

>Note: The AzureChatCompletion service also supports Microsoft Entra authentication. If you don't provide an API key, the service will attempt to authenticate using the Entra token.

In [5]:
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

# Add Azure OpenAI chat completion
chat_completion = AzureChatCompletion(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    service_id="serv-git-chat-1"
)

Let's try chat with LLM model.

In [6]:
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory

execution_settings = AzureChatPromptExecutionSettings()

system_message = """
You are a helpful assistant that can answer questions and provide information.
"""

chat_history = ChatHistory(system_message=system_message)

chat_history.add_user_message("Hello, who are you , what's your model name/version how much token length you can accept from Azure deployment?")

response = await chat_completion.get_chat_message_content(
    chat_history=chat_history,
    settings=execution_settings,
)

print(response)

Hello! I’m an AI language model designed to assist you with a wide range of tasks and questions. I am based on OpenAI’s GPT-4 architecture. When deployed through Microsoft Azure’s OpenAI Service, my model name could appear as `gpt-4`, `gpt-4-32k`, or a similar identifier depending on your deployment.

**Token Length:**
- **Standard GPT-4 (`gpt-4`)**: Accepts up to 8,192 tokens per request (including input and output).
- **Extended GPT-4 (`gpt-4-32k`)**: Accepts up to 32,768 tokens per request (including input and output).

If you’re not sure which model your Azure deployment uses, you can check the resource configuration or ask your Azure administrator.

Let me know if you need more specifics or help with Azure OpenAI Service!


## Creating a Chat Completion Agent

A chat completion agent is fundamentally based on an AI services. As such, creating an chat completion agent starts with creating a Kernel instance that contains one or more chat-completion services and then instantiating the agent with a reference to that Kernel instance.

In [7]:
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings

# Define the Kernel
kernel = Kernel()

# Add the AzureChatCompletion AI Service to the Kernel
kernel.add_service(chat_completion)

settings = AzureChatPromptExecutionSettings(service_id="serv-git-chat-1")

# Create the agent
agent = ChatCompletionAgent(
    kernel=kernel, 
    name="chat-agent", 
    instructions="You are a helpful agent.",
    arguments=KernelArguments(settings=settings)
)

## Conversing with ChatCompletionAgent

There are multiple ways to converse with a ChatCompletionAgent.

The easiest is to call and await get_response:

In [8]:
from semantic_kernel.contents.chat_message_content import ChatMessageContent
from semantic_kernel.contents import AuthorRole

# Define the chat history
history = ChatHistory()

# Add the user message
history.add_message(ChatMessageContent(role=AuthorRole.USER, content="Hello, how are you?"))

# Generate the agent response
response = await agent.get_response(messages=history,arguments=KernelArguments(settings=settings))

print(f"{response.content}")

Hello! I'm just a virtual assistant, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


## Creating a Semantic Kernel Plugin

Let's create a plugin that interacts with Github API for repositories and users.

This plugin example was provided by Microsoft at this link: https://github.com/microsoft/semantic-kernel/blob/main/python/samples/learn_resources/plugins/GithubPlugin/github.py

In [18]:
# Copyright (c) Microsoft. All rights reserved.


import httpx
from pydantic import BaseModel, Field

from semantic_kernel.functions.kernel_function_decorator import kernel_function

# region GitHub Models


class Repo(BaseModel):
    id: int = Field(..., alias="id")
    name: str = Field(..., alias="full_name")
    description: str | None = Field(default=None, alias="description")
    url: str = Field(..., alias="html_url")


class User(BaseModel):
    id: int = Field(..., alias="id")
    login: str = Field(..., alias="login")
    name: str | None = Field(default=None, alias="name")
    company: str | None = Field(default=None, alias="company")
    url: str = Field(..., alias="html_url")


class Label(BaseModel):
    id: int = Field(..., alias="id")
    name: str = Field(..., alias="name")
    description: str | None = Field(default=None, alias="description")


class Issue(BaseModel):
    id: int = Field(..., alias="id")
    number: int = Field(..., alias="number")
    url: str = Field(..., alias="html_url")
    title: str = Field(..., alias="title")
    state: str = Field(..., alias="state")
    labels: list[Label] = Field(..., alias="labels")
    when_created: str | None = Field(default=None, alias="created_at")
    when_closed: str | None = Field(default=None, alias="closed_at")


class IssueDetail(Issue):
    body: str | None = Field(default=None, alias="body")


# endregion


class GitHubSettings(BaseModel):
    base_url: str = "https://api.github.com"
    token: str


class GitHubPlugin:
    def __init__(self, settings: GitHubSettings):
        self.settings = settings

    @kernel_function
    async def get_user_profile(self) -> "User":
        async with self.create_client() as client:
            response = await self.make_request(client, "/user")
            return User(**response)

    @kernel_function
    async def get_repository(self, organization: str, repo: str) -> "Repo":
        async with self.create_client() as client:
            response = await self.make_request(client, f"/repos/{organization}/{repo}")
            return Repo(**response)

    @kernel_function
    async def get_issues(
        self,
        organization: str,
        repo: str,
        max_results: int | None = None,
        state: str = "",
        label: str = "",
        assignee: str = "",
    ) -> list["Issue"]:
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/issues?"
            path = self.build_query(path, "state", state)
            path = self.build_query(path, "assignee", assignee)
            path = self.build_query(path, "labels", label)
            path = self.build_query(path, "per_page", str(max_results) if max_results else "")
            response = await self.make_request(client, path)
            return [Issue(**issue) for issue in response]
    
    @kernel_function
    async def get_commits(
        self,
        organization: str,
        repo: str,
        max_results: int | None = None,
        author: str = "",
        since: str = "",
        until: str = "",
    ) -> list[dict]:
        """
        Retrieve commits from a GitHub repository.

        Args:
            organization (str): The organization or user name.
            repo (str): The repository name.
            max_results (int, optional): Maximum number of commits to return.
            author (str, optional): Filter by commit author.
            since (str, optional): Only commits after this date (ISO 8601).
            until (str, optional): Only commits before this date (ISO 8601).

        Returns:
            list[dict]: A list of commit data dictionaries.
        """
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/commits?"
            path = self.build_query(path, "author", author)
            path = self.build_query(path, "since", since)
            path = self.build_query(path, "until", until)
            path = self.build_query(path, "per_page", str(max_results) if max_results else "")
            response = await self.make_request(client, path)
            return response
        
    @kernel_function
    async def get_commit_detail(
        self,
        organization: str,
        repo: str,
        commit_sha: str,
    ) -> dict:
        """
        Retrieve details for a specific commit in a GitHub repository.

        Args:
            organization (str): The organization or user name.
            repo (str): The repository name.
            commit_sha (str): The commit SHA.

        Returns:
            dict: The commit details.
        """
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/commits/{commit_sha}"
            response = await self.make_request(client, path)
            return response
        
    @kernel_function
    async def get_commit_diff(
        self,
        organization: str,
        repo: str,
        base_commit: str,
        head_commit: str,
    ) -> dict:
        """
        Retrieve the diff (code changes) between two commits in a GitHub repository.

        Args:
            organization (str): The organization or user name.
            repo (str): The repository name.
            base_commit (str): The base commit SHA.
            head_commit (str): The head commit SHA.

        Returns:
            dict: The comparison result including files changed, commits, and diff stats.
        """
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/compare/{base_commit}...{head_commit}"
            response = await self.make_request(client, path)
            return response
        
    @kernel_function
    async def create_git_issue_with_labels(
        self,
        organization: str,
        repo: str,
        title: str,
        body: str ,
        labels: list[str]
    ) -> dict:
        """
        Create a new issue with dummy text and labels in the specified repository.

        Args:
            organization (str): The organization or user name.
            repo (str): The repository name.
            title (str, optional): The title of the issue.
            body (str, optional): The body content of the issue.
            labels (list[str], optional): List of labels to assign.

        Returns:
            dict: The created issue details.
        """
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/issues"
            payload = {"title": title, "body": body, "labels": labels}
            print(f"POST REQUEST: {path}\nPayload: {payload}")
            response = await client.post(path, json=payload)
            response.raise_for_status()
            return response.json()

    @kernel_function
    async def get_issue_detail(self, organization: str, repo: str, issue_id: int) -> "IssueDetail":
        async with self.create_client() as client:
            path = f"/repos/{organization}/{repo}/issues/{issue_id}"
            response = await self.make_request(client, path)
            return IssueDetail(**response)

    def create_client(self) -> httpx.AsyncClient:
        headers = {
            "User-Agent": "request",
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {self.settings.token}",
            "X-GitHub-Api-Version": "2022-11-28",
        }
        return httpx.AsyncClient(base_url=self.settings.base_url, headers=headers, timeout=5)

    @staticmethod
    def build_query(path: str, key: str, value: str) -> str:
        if value:
            return f"{path}{key}={value}&"
        return path

    @staticmethod
    async def make_request(client: httpx.AsyncClient, path: str) -> dict:
        print(f"REQUEST: {path}\n")
        response = await client.get(path)
        response.raise_for_status()
        return response.json()

## Agent Definition

Finally we are ready to instantiate a ChatCompletionAgent with its Instructions, associated Kernel, and the default Arguments and Execution Settings. In this case, we desire to have the any plugin functions automatically executed.

>You will need to define settings for either OpenAI or Azure OpenAI and also for GitHub.

>Note: For information on GitHub Personal Access Tokens, see: [Managing your personal access tokens](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In [19]:
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.kernel import Kernel

# Import GitHubPlugin and GitHubSettings directly from the notebook
from __main__ import GitHubPlugin, GitHubSettings

from jinja2 import Template

def update_agent_instructions(agent, **kwargs):
    agent.instructions = inst_template.render(**kwargs)
    return agent

inst_template = f"""You are an agent designed to query and retrieve information from a single GitHub repository in a read-only manner.
        You are also able to access the profile of the active user.
        Use the current date and time to provide up-to-date details or time-sensitive responses.
        Use the values from arguments:
        - The repository name: {{repo_name}}
        - The current datetime: {{now}}
"""
kernel = Kernel()
# Add the AzureChatCompletion AI Service to the Kernel
service_id = "serv-git-chat-1"
kernel.add_service(AzureChatCompletion(service_id=service_id))

settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
# Configure the function choice behavior to auto invoke kernel functions
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# Set your GitHub Personal Access Token (PAT) value here
gh_settings = GitHubSettings(token=os.getenv("GITHUB_PAT"))

kernel.add_plugin(plugin=GitHubPlugin(gh_settings), plugin_name="GithubPlugin")

# Create the agent
agent = ChatCompletionAgent(
    kernel=kernel,
    name="GitAssistantAgent",
    instructions=inst_template,
    arguments=KernelArguments(settings=settings,repo_name="harsha3187/git-dev-agent"),
)

Let's get the username of the current user.

In [32]:
from semantic_kernel.contents.chat_message_content import ChatMessageContent
from datetime import datetime

chat_history = ChatHistory()

user_input = "What is my username?"

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

async for response in agent.invoke(messages=chat_history):
    print(response.content)

REQUEST: /user

Your GitHub username is harsha3187.


In [36]:

chat_history.add_message(ChatMessageContent(role=AuthorRole.ASSISTANT, content=user_input))


In [35]:
user_input = "Describe the repo "

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(repo_name="harsha3187/git-dev-agent",now=datetime.now().strftime("%Y-%m-%d %H:%M"))

response = await kernel.invoke(messages=chat_history, arguments=arguments)

# If the response is iterable, iterate over it
if hasattr(response, '__aiter__'):
    async for item in response:
        print(item.content)
else:
    print(response.content)

KernelFunctionNotFoundError: No function, or function name and plugin name provided

Describe the repo.

Describe the newest issue created in the repo.

In [12]:
user_input = "Create a issue on wrong formatting in git_plugin.py file labelled under Formatting, bug."

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /user

REQUEST: /repos/harsha3187/git-dev-agent

POST REQUEST: /repos/harsha3187/git-dev-agent/issues
Payload: {'title': 'Wrong formatting in git_plugin.py file', 'body': 'There is an issue with the formatting in the git_plugin.py file. Please review and fix the formatting issues to maintain code consistency and readability.', 'labels': ['Formatting', 'bug']}
Here are the answers to your queries:

1. Your username is harsha3187.

2. Repository Description:
   - Name: harsha3187/git-dev-agent
   - Description: Git agent using semantic kernel
   - URL: https://github.com/harsha3187/git-dev-agent

3. An issue has been created regarding wrong formatting in the git_plugin.py file labelled under Formatting and bug. You can view it here: https://github.com/harsha3187/git-dev-agent/issues/2


In [26]:
user_input = "Describe the newest issue created in the repo."

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python/issues?state=all&per_page=1&

Repository Description:
Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python is a Multi-Agent AI Application (Python) that leverages Semantic-Kernel along with Azure AI Agent Service in Azure AI Foundry. The project integrates advanced AI agent capabilities with Azure's services for creating multi-agent solutions.

Newest Issue:
- Title: AttributeError occurred: 'dict' object has no attribute 'metadata'
- Status: Open
- Created: 2025-04-13
- Link: View issue on GitHub

Let me know if you want more details about the issue or any other aspect of the repository!


List the top 10 issues closed within the last week.

In [27]:
user_input = "List the top 10 issues closed within the last week."

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python/issues?state=closed&per_page=20&

There have been no issues closed within the last week in the repository Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python. If you need information about older issues or open issues, let me know!


How were these issues labeled?

In [28]:
user_input = "How were these issues labeled?"

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python/issues?state=closed&per_page=20&

There are no issues that were closed within the last week in the repository Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python. Therefore, no labels were applied to any recently closed issues.


List the 5 most recently opened issues with the "Bug" label.

In [29]:
user_input = "List the 5 most recently opened issues with the 'Bug' label."

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python/issues?state=open&labels=Bug&per_page=20&

There are currently no open issues labeled 'Bug' in the repository "Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python." If you need details about closed 'Bug' issues or would like a broader search, please let me know!


In [35]:
user_input = "What changed from Issue#412 to Issue#413?"

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

I'm sorry for the confusion, but as an agent, I'm designed to read information from a GitHub repository. I currently don't have the capability to compare different issues or pull requests and identify code changes between them. You may need to manually check the associated pull requests or code changes linked with individual issue numbers.


In [30]:
user_input = "Give me the last 5 commits with brief descriptions?"

chat_history.add_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

arguments = KernelArguments(now=datetime.now().strftime("%Y-%m-%d %H:%M"))

async for response in agent.invoke(messages=chat_history, arguments=arguments):
    print(f"{response.content}")

REQUEST: /repos/Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python/commits?per_page=5&

Here are the last 5 commits for the repo "Shailender-Youtube/Multi-Agent-Semantic-Kernel-Azure-Agent-Service-Python" with brief descriptions:

1. **Update README.md**
   - SHA: cb5ed94446b3ed0a8df1e45296737172de6f32fd
   - Author: schoudhary22
   - Date: 2025-03-06
   - Changes made to update the README file.

2. **Add files via upload**
   - SHA: abb52e7709c6a60e2f1eb463a02ffd0beb2d725a
   - Author: schoudhary22
   - Date: 2025-03-06
   - Uploaded additional files to the repository.

3. **Initial commit**
   - SHA: bd5d5f512492d649f51c04bbb94a9708c8ee2e42
   - Author: schoudhary22
   - Date: 2025-03-06
   - The initial commit, setting up the repository.

There are only 3 commits currently in the repository. If you need more details or information about any specific commit, let me know!


## More resources

https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/examples/example-chat-agent?pivots=programming-language-python

In [20]:
# Use the plugin's registered function to invoke get_commit_diff
commit_diff = await plugin.functions["get_commits"].method(
	organization="microsoft",
	repo="document-generation-solution-accelerator")

print(commit_diff)

REQUEST: /repos/microsoft/document-generation-solution-accelerator/commits?

[{'sha': '6437951b7f57dd95365813be0920890c7fcff3a7', 'node_id': 'C_kwDOMOAjG9oAKDY0Mzc5NTFiN2Y1N2RkOTUzNjU4MTNiZTA5MjA4OTBjN2ZjZmYzYTc', 'commit': {'author': {'name': 'Roopan-Microsoft', 'email': '168007406+Roopan-Microsoft@users.noreply.github.com', 'date': '2025-05-06T04:32:27Z'}, 'committer': {'name': 'GitHub', 'email': 'noreply@github.com', 'date': '2025-05-06T04:32:27Z'}, 'message': 'Merge pull request #426 from microsoft/psl-codeowners\n\nchore: Update CODEOWNERS', 'tree': {'sha': 'ba579189ee885d7e410d9a97045f2f9e811b4cd7', 'url': 'https://api.github.com/repos/microsoft/document-generation-solution-accelerator/git/trees/ba579189ee885d7e410d9a97045f2f9e811b4cd7'}, 'url': 'https://api.github.com/repos/microsoft/document-generation-solution-accelerator/git/commits/6437951b7f57dd95365813be0920890c7fcff3a7', 'comment_count': 0, 'verification': {'verified': True, 'reason': 'valid', 'signature': '-----BEGIN PGP